═════════════════════════════════════════════════════════════════════════════               
            **QUANTITATIVE TRADING STRATEGY ENGINE**                              
            Trend Detection · Multi-Strategy · Gain-Optimized Signals         
═════════════════════════════════════════════════════════════════════════════

Architecture:
  1. DataEngine       — loads & preprocesses fundamental data
  2. FeatureEngine    — computing multiple technical indicators
  3. TrendClassifier  — detects UP / DOWN / SIDEWAYS regime based on several algorithm's result, ensemble method has been used to finalize the trend.
  4. StrategyLibrary  — 9 strategies, each tuned for one or more regimes
  5. StrategySelector — ranks strategies by regime fit + recent performance
  6. Backtester       — vectorised backtest with commissions, slippage, stops
  7. RiskManager      — position sizing, stop-loss, take-profit, max drawdown gate
  8. SignalEngine     — consensus signal with confidence score
  9. Reporter         — full performance breakdown + trade log

By Sri Gokul Prazath



### Import & Installation

In [1]:
# !pip install pandas_ta

In [2]:
import warnings
warnings.filterwarnings("ignore")
import pandas_ta as ta
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from enum import Enum
import math

### Variable & Data Declaration

In [3]:
class Regime(Enum):
    UP       = "UPTREND"
    DOWN     = "DOWNTREND"
    SIDEWAYS = "SIDEWAYS"

class Signal(Enum):
    BUY      = "BUY"
    SELL     = "SELL"
    HOLD     = "HOLD"

COMMISSION = 0.001          # 0.1% per side
SLIPPAGE   = 0.0005         # 0.05% slippage estimate
RISK_PER_TRADE = 0.02       # max 2% portfolio risk per trade
MAX_DRAWDOWN_GATE = -0.15   # halt trading if drawdown > 15%


In [4]:
class DataEngine:
    """
    Accepts either:
      - A pandas DataFrame with columns: Open, High, Low, Close, Volume
      - A dict of fundamental data (optional overlay)
    Validates, cleans, and prepares the dataset.
    """

    REQUIRED_COLS = ["Open", "High", "Low", "Close", "Volume"]

    def __init__(self, price_df: pd.DataFrame, fundamentals: Optional[Dict] = None):
        self.raw       = price_df.copy()
        self.fundamentals = fundamentals or {}
        self.data: pd.DataFrame = pd.DataFrame()

    def load(self) -> "DataEngine":
        df = self.raw.copy()

        # normalise column names
        df.columns = [c.strip().title() for c in df.columns]
        missing = [c for c in self.REQUIRED_COLS if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")

        df = df[self.REQUIRED_COLS].copy()
        df = df.apply(pd.to_numeric, errors="coerce")
        df.dropna(inplace=True)

        # basic sanity
        df = df[df["Close"] > 0]
        df = df[df["Volume"] > 0]
        df.index = pd.to_datetime(df.index)
        df.sort_index(inplace=True)

        self.data = df
        return self

    def get(self) -> pd.DataFrame:
        return self.data.copy()

    def summary(self) -> Dict:
        df = self.data
        return {
            "rows":       len(df),
            "start":      str(df.index[0].date()),
            "end":        str(df.index[-1].date()),
            "price_last": round(df["Close"].iloc[-1], 2),
            "price_high": round(df["High"].max(), 2),
            "price_low":  round(df["Low"].min(), 2),
            "avg_volume": int(df["Volume"].mean()),
        }



### Feature Indicators

In [5]:
class FeatureEngine:
    """Computes all technical indicators in-place on a DataFrame."""

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    # ── helpers ──────────────────────────────────────────────────────────────
    @staticmethod
    def _ema(series: pd.Series, n: int) -> pd.Series:
        return series.ewm(span=n, adjust=False).mean()

    @staticmethod
    def _rma(series: pd.Series, n: int) -> pd.Series:
        return series.ewm(alpha=1 / n, adjust=False).mean()

    # ── trend indicators ─────────────────────────────────────────────────────
    def add_moving_averages(self):
        c = self.df["Close"]
        for n in [5, 10, 20, 50, 100, 200]:
            self.df[f"SMA_{n}"]  = c.rolling(n).mean()
            self.df[f"EMA_{n}"]  = self._ema(c, n)
        # VWAP (daily approximation)
        self.df["VWAP"] = (self.df["Close"] * self.df["Volume"]).cumsum() / self.df["Volume"].cumsum()
        return self

    def add_macd(self, fast=12, slow=26, signal=9):
        c = self.df["Close"]
        ema_fast = self._ema(c, fast)
        ema_slow = self._ema(c, slow)
        self.df["MACD"]        = ema_fast - ema_slow
        self.df["MACD_Signal"] = self._ema(self.df["MACD"], signal)
        self.df["MACD_Hist"]   = self.df["MACD"] - self.df["MACD_Signal"]
        return self

    def add_adx(self, n=14):
        h, l, c = self.df["High"], self.df["Low"], self.df["Close"]
        prev_c  = c.shift(1)
        tr      = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
        dm_pos  = np.where((h - h.shift(1)) > (l.shift(1) - l), np.maximum(h - h.shift(1), 0), 0)
        dm_neg  = np.where((l.shift(1) - l) > (h - h.shift(1)), np.maximum(l.shift(1) - l, 0), 0)
        atr     = self._rma(tr, n)
        di_pos  = 100 * self._rma(pd.Series(dm_pos, index=c.index), n) / atr
        di_neg  = 100 * self._rma(pd.Series(dm_neg, index=c.index), n) / atr
        dx      = 100 * (di_pos - di_neg).abs() / (di_pos + di_neg + 1e-9)
        self.df["ADX"]    = self._rma(dx, n)
        self.df["DI_pos"] = di_pos
        self.df["DI_neg"] = di_neg
        return self

    # ── momentum indicators ──────────────────────────────────────────────────
    def add_rsi(self, n=14):
        delta = self.df["Close"].diff()
        gain  = delta.clip(lower=0)
        loss  = (-delta).clip(lower=0)
        avg_g = self._rma(gain, n)
        avg_l = self._rma(loss, n)
        rs    = avg_g / (avg_l + 1e-9)
        self.df["RSI"] = 100 - (100 / (1 + rs))
        return self

    def add_stochastic(self, k=14, d=3):
        lo  = self.df["Low"].rolling(k).min()
        hi  = self.df["High"].rolling(k).max()
        k_  = 100 * (self.df["Close"] - lo) / (hi - lo + 1e-9)
        self.df["Stoch_K"] = k_
        self.df["Stoch_D"] = k_.rolling(d).mean()
        return self

    def add_cci(self, n=20):
        tp = (self.df["High"] + self.df["Low"] + self.df["Close"]) / 3
        ma = tp.rolling(n).mean()
        md = tp.rolling(n).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True)
        self.df["CCI"] = (tp - ma) / (0.015 * md + 1e-9)
        return self

    def add_mfi(self, n=14):
        tp  = (self.df["High"] + self.df["Low"] + self.df["Close"]) / 3
        rmf = tp * self.df["Volume"]
        pos = rmf.where(tp > tp.shift(1), 0)
        neg = rmf.where(tp <= tp.shift(1), 0)
        mfr = pos.rolling(n).sum() / (neg.rolling(n).sum() + 1e-9)
        self.df["MFI"] = 100 - (100 / (1 + mfr))
        return self

    # ── volatility indicators ────────────────────────────────────────────────
    def add_bollinger(self, n=20, k=2):
        ma  = self.df["Close"].rolling(n).mean()
        sd  = self.df["Close"].rolling(n).std()
        self.df["BB_Upper"] = ma + k * sd
        self.df["BB_Lower"] = ma - k * sd
        self.df["BB_Mid"]   = ma
        self.df["BB_Width"] = (self.df["BB_Upper"] - self.df["BB_Lower"]) / (ma + 1e-9)
        self.df["BB_pct"]   = (self.df["Close"] - self.df["BB_Lower"]) / (self.df["BB_Upper"] - self.df["BB_Lower"] + 1e-9)
        return self

    def add_atr(self, n=14):
        h, l, c = self.df["High"], self.df["Low"], self.df["Close"]
        tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
        self.df["ATR"] = self._rma(tr, n)
        self.df["ATR_pct"] = self.df["ATR"] / self.df["Close"]
        return self

    def add_keltner(self, n=20, mult=2):
        mid = self._ema(self.df["Close"], n)
        atr = self.df["ATR"] if "ATR" in self.df.columns else self.df["High"] - self.df["Low"]
        self.df["KC_Upper"] = mid + mult * atr
        self.df["KC_Lower"] = mid - mult * atr
        return self

    # ── volume indicators ────────────────────────────────────────────────────
    def add_obv(self):
        direction = np.sign(self.df["Close"].diff())
        self.df["OBV"] = (self.df["Volume"] * direction).cumsum()
        return self

    def add_volume_sma(self, n=20):
        self.df["VOL_SMA"]   = self.df["Volume"].rolling(n).mean()
        self.df["VOL_ratio"] = self.df["Volume"] / (self.df["VOL_SMA"] + 1e-9)
        return self

    def add_cmf(self, n=20):
        mfv = ((self.df["Close"] - self.df["Low"]) - (self.df["High"] - self.df["Close"])) / \
              (self.df["High"] - self.df["Low"] + 1e-9) * self.df["Volume"]
        self.df["CMF"] = mfv.rolling(n).sum() / (self.df["Volume"].rolling(n).sum() + 1e-9)
        return self

    # ── price structure ──────────────────────────────────────────────────────
    def add_returns(self):
        c = self.df["Close"]
        self.df["Return_1d"]  = c.pct_change(1)
        self.df["Return_5d"]  = c.pct_change(5)
        self.df["Return_20d"] = c.pct_change(20)
        self.df["Volatility"] = self.df["Return_1d"].rolling(20).std() * np.sqrt(252)
        return self

    def add_support_resistance(self, n=20):
        self.df["Pivot"]    = (self.df["High"] + self.df["Low"] + self.df["Close"]) / 3
        self.df["R1"]       = 2 * self.df["Pivot"] - self.df["Low"]
        self.df["S1"]       = 2 * self.df["Pivot"] - self.df["High"]
        self.df["R2"]       = self.df["Pivot"] + (self.df["High"] - self.df["Low"])
        self.df["S2"]       = self.df["Pivot"] - (self.df["High"] - self.df["Low"])
        self.df["High_N"]   = self.df["High"].rolling(n).max()
        self.df["Low_N"]    = self.df["Low"].rolling(n).min()
        return self

    def add_trend_strength(self):
        """Composite trend-strength score: range 0-100."""
        df = self.df
        score = pd.Series(50.0, index=df.index)
        if "ADX" in df:
            score += (df["ADX"].clip(0, 50) - 25)               # ADX contribution
        if "MACD_Hist" in df:
            norm = df["MACD_Hist"] / (df["Close"] + 1e-9) * 100
            score += norm.clip(-10, 10)
        self.df["Trend_Score"] = score.clip(0, 100)
        return self

    def build_all(self) -> pd.DataFrame:
        (self
         .add_moving_averages()
         .add_macd()
         .add_rsi()
         .add_stochastic()
         .add_cci()
         .add_mfi()
         .add_bollinger()
         .add_atr()
         .add_keltner()
         .add_obv()
         .add_volume_sma()
         .add_cmf()
         .add_returns()
         .add_support_resistance()
         .add_adx()
         .add_trend_strength()
        )
        self.df.dropna(inplace=True)
        return self.df


### Trend Identifiers

In [6]:
class TrendClassifier:
    """
    Five independent classifiers vote on regime:
      1. MA Slope   — slope of 50-day SMA
      2. ADX/DI     — trend strength + direction
      3. BB Width   — low width = sideways
      4. Price vs MA— price relative to 20/50/200 EMA
      5. Linear Reg — slope of 20-day linear regression on close
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df

    def _ma_slope_vote(self) -> Regime:
        sma = self.df["SMA_50"].dropna()
        if len(sma) < 10:
            return Regime.SIDEWAYS
        slope = (sma.iloc[-1] - sma.iloc[-10]) / sma.iloc[-10]
        if slope >  0.01:  return Regime.UP
        if slope < -0.01:  return Regime.DOWN
        return Regime.SIDEWAYS

    def _adx_vote(self) -> Regime:
        row = self.df.iloc[-1]
        adx, di_p, di_n = row.get("ADX", 20), row.get("DI_pos", 20), row.get("DI_neg", 20)
        if adx < 15:  # FIX 5: lowered threshold; real stocks rarely have ADX>20
            return Regime.SIDEWAYS
        if di_p > di_n:
            return Regime.UP
        return Regime.DOWN

    def _bb_width_vote(self) -> Regime:
        bw  = self.df["BB_Width"].dropna()
        if len(bw) < 20:
            return Regime.SIDEWAYS
        recent = bw.iloc[-1]
        hist   = bw.rolling(50).mean().iloc[-1]
        if recent < hist * 0.75:
            return Regime.SIDEWAYS
        c   = self.df["Close"].iloc[-1]
        mid = self.df["BB_Mid"].iloc[-1]
        return Regime.UP if c > mid else Regime.DOWN

    def _price_ma_vote(self) -> Regime:
        row = self.df.iloc[-1]
        c   = row["Close"]
        e20 = row.get("EMA_20", c)
        e50 = row.get("EMA_50", c)
        e200= row.get("EMA_200", c)
        score = sum([c > e20, c > e50, c > e200, e20 > e50, e50 > e200])
        if score >= 4:  return Regime.UP
        if score <= 1:  return Regime.DOWN
        return Regime.SIDEWAYS

    def _linreg_vote(self) -> Regime:
        close = self.df["Close"].tail(20).values
        if len(close) < 10:
            return Regime.SIDEWAYS
        x     = np.arange(len(close))
        slope, _ = np.polyfit(x, close, 1)
        rel   = slope / (close.mean() + 1e-9)
        if rel >  0.003:  return Regime.UP
        if rel < -0.003:  return Regime.DOWN
        return Regime.SIDEWAYS

    def classify(self) -> Tuple[Regime, float, Dict]:
        votes = [
            self._ma_slope_vote(),
            self._adx_vote(),
            self._bb_width_vote(),
            self._price_ma_vote(),
            self._linreg_vote(),
        ]
        tally = {r: 0 for r in Regime}
        for v in votes:
            tally[v] += 1

        winner     = max(tally, key=tally.get)
        confidence = tally[winner] / len(votes)        # 0.4 – 1.0

        detail = {
            "MA Slope":      votes[0].value,
            "ADX/DI":        votes[1].value,
            "BB Width":      votes[2].value,
            "Price vs MA":   votes[3].value,
            "Linear Reg":    votes[4].value,
            "Votes UP":      tally[Regime.UP],
            "Votes DOWN":    tally[Regime.DOWN],
            "Votes SIDEWAYS":tally[Regime.SIDEWAYS],
        }
        return winner, confidence, detail


### Defining Strategy

In [7]:
@dataclass
class StrategyResult:
    name:           str
    regime_fit:     List[Regime]
    signals:        pd.Series       # +1 buy, -1 sell, 0 hold
    description:    str = ""
    param_summary:  str = ""


In [8]:

class StrategyLibrary:
    """
    Each strategy returns a StrategyResult with a signals Series.
    All strategies include:
      - Entry filter (regime check)
      - ATR-based stop-loss
      - Confirmation via volume / momentum
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    # ── TREND STRATEGIES ─────────────────────────────────────────────────────

    def strategy_golden_cross(self) -> StrategyResult:
        """EMA 20/50 crossover with ADX > 20 confirmation."""
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        fast = df["EMA_20"]
        slow = df["EMA_50"]
        adx  = df.get("ADX", pd.Series(25, index=df.index))
        vol  = df.get("VOL_ratio", pd.Series(1, index=df.index))

        buy_cond  = (fast.shift(1) < slow.shift(1)) & (fast >= slow) & (adx > 20) & (vol > 0.8)
        sell_cond = (fast.shift(1) > slow.shift(1)) & (fast <= slow)

        sig[buy_cond]  =  1
        sig[sell_cond] = -1
        return StrategyResult("Golden Cross EMA 20/50", [Regime.UP],
                              sig, "EMA 20/50 golden/death cross with ADX>20 and volume filter.",
                              "EMA(20), EMA(50), ADX>20, VOL>0.8x avg")

    def strategy_macd_trend(self) -> StrategyResult:
        """MACD histogram sign change + RSI not overbought/oversold."""
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        hist = df["MACD_Hist"]
        rsi  = df["RSI"]

        buy_cond  = (hist.shift(1) < 0) & (hist >= 0) & (rsi < 65) & (rsi > 40)
        sell_cond = (hist.shift(1) > 0) & (hist <= 0) & (rsi > 35) & (rsi < 60)

        sig[buy_cond]  =  1
        sig[sell_cond] = -1
        return StrategyResult("MACD Trend Filter", [Regime.UP, Regime.DOWN],
                              sig, "MACD histogram sign flip + RSI regime filter.",
                              "MACD(12,26,9), RSI(14) 40-65 filter")

    def strategy_triple_ma(self) -> StrategyResult:
        """Triple MA: price > EMA20 > EMA50 > EMA200 full alignment."""
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        c    = df["Close"]
        e20  = df["EMA_20"]
        e50  = df["EMA_50"]
        e200 = df["EMA_200"]

        bull_align  = (c > e20) & (e20 > e50) & (e50 > e200)
        bear_align  = (c < e20) & (e20 < e50) & (e50 < e200)
        prev_bull   = bull_align.shift(1).fillna(False)
        prev_bear   = bear_align.shift(1).fillna(False)

        sig[bull_align & ~prev_bull] =  1
        sig[bear_align & ~prev_bear] = -1
        sig[~bull_align & ~bear_align & prev_bull] = -1   # exit bull
        return StrategyResult("Triple MA Alignment", [Regime.UP],
                              sig, "Full EMA 20/50/200 bullish alignment signal.",
                              "EMA(20), EMA(50), EMA(200)")

    # ── MOMENTUM STRATEGIES ──────────────────────────────────────────────────

    def strategy_rsi_momentum(self) -> StrategyResult:
        """RSI breakout from neutral zone with trend confirmation."""
        df  = self.df
        sig = pd.Series(0, index=df.index)
        rsi = df["RSI"]
        c   = df["Close"]
        e50 = df["EMA_50"]

        buy_cond  = (rsi.shift(1) < 55) & (rsi >= 55) & (c > e50)   # RSI breakout up
        sell_cond = (rsi.shift(1) > 45) & (rsi <= 45) & (c < e50)   # RSI break down

        sig[buy_cond]  =  1
        sig[sell_cond] = -1
        return StrategyResult("RSI Momentum Breakout", [Regime.UP, Regime.DOWN],
                              sig, "RSI crossing 55/45 with price-trend filter.",
                              "RSI(14) cross 55/45, price vs EMA50")

    def strategy_stochastic_swing(self) -> StrategyResult:
        """Stochastic oversold/overbought for swing entries."""
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        k    = df["Stoch_K"]
        d    = df["Stoch_D"]

        buy_cond  = (k.shift(1) < 25) & (k >= 25) & (k > d)
        sell_cond = (k.shift(1) > 75) & (k <= 75) & (k < d)

        sig[buy_cond]  =  1
        sig[sell_cond] = -1
        return StrategyResult("Stochastic Swing", [Regime.SIDEWAYS, Regime.UP],
                              sig, "Stochastic %K/%D oversold bounce / overbought fade.",
                              "Stoch K(14), D(3), levels 25/75")

    # ── MEAN REVERSION STRATEGIES ────────────────────────────────────────────

    def strategy_bollinger_reversion(self) -> StrategyResult:
        """Price touches lower/upper BB in a confirmed range environment."""
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        c    = df["Close"]
        bbl  = df["BB_Lower"]
        bbu  = df["BB_Upper"]
        bbm  = df["BB_Mid"]
        bbw  = df["BB_Width"]
        bbw_avg = bbw.rolling(50).mean()
        rsi  = df["RSI"]

        # Only fire in narrow-bandwidth (sideways / low-vol) regimes
        sideways = bbw < bbw_avg * 1.1

        buy_cond  = (c.shift(1) <= bbl.shift(1)) & (c > bbl) & sideways & (rsi < 45)
        sell_cond = (c.shift(1) >= bbu.shift(1)) & (c < bbu) & sideways & (rsi > 55)

        sig[buy_cond]  =  1
        sig[sell_cond] = -1
        return StrategyResult("Bollinger Band Reversion", [Regime.SIDEWAYS],
                              sig, "Mean-reversion on BB touch, only in low-bandwidth regime.",
                              "BB(20,2), BB Width < avg, RSI filter")

    def strategy_rsi_oversold(self) -> StrategyResult:
        """Classic RSI oversold/overbought with volume spike confirmation."""
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        rsi  = df["RSI"]
        vol  = df.get("VOL_ratio", pd.Series(1, index=df.index))

        buy_cond  = (rsi.shift(1) < 30) & (rsi >= 30) & (vol > 1.2)
        sell_cond = (rsi.shift(1) > 70) & (rsi <= 70)

        sig[buy_cond]  =  1
        sig[sell_cond] = -1
        return StrategyResult("RSI Oversold/Overbought", [Regime.SIDEWAYS, Regime.DOWN],
                              sig, "RSI 30/70 extremes + 1.2x average volume spike.",
                              "RSI(14) 30/70, Volume > 1.2x avg")

    # ── BREAKOUT STRATEGY ────────────────────────────────────────────────────

    def strategy_donchian_breakout(self) -> StrategyResult:
        """N-day high breakout with volume and ATR expansion."""
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        c    = df["Close"]
        hi   = df["High"].rolling(20).max().shift(1)
        lo   = df["Low"].rolling(20).min().shift(1)
        vol  = df.get("VOL_ratio", pd.Series(1, index=df.index))
        atr  = df.get("ATR_pct", pd.Series(0.01, index=df.index))
        atr_avg = atr.rolling(20).mean()

        buy_cond  = (c > hi) & (vol > 1.5) & (atr > atr_avg * 0.9)
        sell_cond = (c < lo) & (vol > 1.3)

        sig[buy_cond]  =  1
        sig[sell_cond] = -1
        return StrategyResult("Donchian Breakout", [Regime.UP],
                              sig, "20-day high breakout + 1.5x volume + ATR expansion.",
                              "20-day channel, VOL > 1.5x, ATR expansion")

    # ── DOWNTREND STRATEGY ───────────────────────────────────────────────────

    def strategy_bear_reversal(self) -> StrategyResult:
        """
        Identifies exhaustion bounces in a downtrend for short entries
        (or cash-out points for long investors).
        """
        df   = self.df
        sig  = pd.Series(0, index=df.index)
        rsi  = df["RSI"]
        cci  = df.get("CCI", pd.Series(0, index=df.index))
        mfi  = df.get("MFI", pd.Series(50, index=df.index))
        c    = df["Close"]
        e20  = df["EMA_20"]
        e50  = df["EMA_50"]

        # Downtrend: price below both EMAs, bounce to EMA20 = sell signal
        down_regime = (c < e50) & (e20 < e50)
        bounce_sell = down_regime & (c.shift(1) < e20) & (c >= e20) & (rsi > 50) & (mfi < 60)
        # Oversold capitulation = potential cover / buy
        cover_buy   = down_regime & (rsi < 25) & (cci < -150) & (mfi < 20)

        sig[bounce_sell] = -1
        sig[cover_buy]   =  1
        return StrategyResult("Bear Reversal (Downtrend)", [Regime.DOWN],
                              sig, "Sell on oversold bounce in downtrend; buy on capitulation.",
                              "RSI, CCI(-150), MFI(<20), price vs EMA20/50")

    def get_all(self) -> List[StrategyResult]:
        return [
            self.strategy_golden_cross(),
            self.strategy_macd_trend(),
            self.strategy_triple_ma(),
            self.strategy_rsi_momentum(),
            self.strategy_stochastic_swing(),
            self.strategy_bollinger_reversion(),
            self.strategy_rsi_oversold(),
            self.strategy_donchian_breakout(),
            self.strategy_bear_reversal(),
        ]



### Selecting Strategy

In [9]:

class StrategySelector:
    """
    Given detected regime, ranks strategies by:
      - Regime alignment (primary)
      - Recent backtest Sharpe (secondary, last 60 bars)
    Returns top-K strategies.
    """

    def __init__(self, df: pd.DataFrame, regime: Regime, confidence: float):
        self.df         = df
        self.regime     = regime
        self.confidence = confidence

    def _quick_sharpe(self, signals: pd.Series) -> float:
        """Fast approximate Sharpe on last 60 bars."""
        df  = self.df.tail(60)
        sig = signals.reindex(df.index).fillna(0)
        rets= df["Return_1d"].fillna(0)
        strategy_rets = sig.shift(1).fillna(0) * rets
        if strategy_rets.std() < 1e-9:
            return 0.0
        return float(strategy_rets.mean() / strategy_rets.std() * np.sqrt(252))

    def rank(self, strategies: List[StrategyResult], top_k: int = 3) -> List[StrategyResult]:
        scored = []
        for s in strategies:
            regime_fit = 1.0 if self.regime in s.regime_fit else 0.1
            sharpe     = max(self._quick_sharpe(s.signals), 0)     # ignore negative Sharpe
            score      = regime_fit * (1 + 0.3 * sharpe)
            scored.append((score, s))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [s for _, s in scored[:top_k]]



In [10]:

@dataclass
class TradeParams:
    entry_price:    float
    stop_loss:      float
    take_profit:    float
    position_size:  float
    risk_reward:    float


### Risk Manager

In [11]:

class RiskManager:
    """
    ATR-based stop-loss and take-profit.
    Kelly-criterion inspired position sizing (capped at RISK_PER_TRADE).
    Drawdown gate: halts new trades when portfolio drawdown > threshold.
    """

    def __init__(self, df: pd.DataFrame, regime: Regime, portfolio_value: float):
        self.df        = df
        self.regime    = regime
        self.portfolio = portfolio_value

    def compute_trade_params(self, signal: Signal) -> Optional[TradeParams]:
        row  = self.df.iloc[-1]
        c    = row["Close"]
        atr  = row.get("ATR", c * 0.02)

        # Regime-dependent multipliers
        sl_mult = {"UP": 1.5, "DOWN": 1.5, "SIDEWAYS": 1.0}[self.regime.name]
        tp_mult = {"UP": 3.0, "DOWN": 2.5, "SIDEWAYS": 1.8}[self.regime.name]

        if signal == Signal.BUY:
            stop_loss   = c - sl_mult * atr
            take_profit = c + tp_mult * atr
        elif signal == Signal.SELL:
            stop_loss   = c + sl_mult * atr
            take_profit = c - tp_mult * atr
        else:
            return None

        risk_per_share  = abs(c - stop_loss)
        max_risk_dollar = self.portfolio * RISK_PER_TRADE
        position_size   = min(max_risk_dollar / (risk_per_share + 1e-9) * c / self.portfolio, 0.20)
        rr              = abs(take_profit - c) / (risk_per_share + 1e-9)

        return TradeParams(
            entry_price   = round(c, 4),
            stop_loss     = round(stop_loss, 4),
            take_profit   = round(take_profit, 4),
            position_size = round(position_size, 4),
            risk_reward   = round(rr, 2),
        )


In [12]:
@dataclass
class BacktestResult:
    strategy_name:    str
    final_value:      float
    total_return_pct: float
    sharpe_ratio:     float
    max_drawdown_pct: float
    win_rate_pct:     float
    total_trades:     int
    profit_factor:    float
    calmar_ratio:     float
    trades:           List[Dict] = field(default_factory=list)
    portfolio_series: List[float] = field(default_factory=list)


### BackTesting

In [13]:
class Backtester:
    """
    Realistic backtester with:
      - Commission + slippage on every fill
      - ATR-based trailing stop-loss
      - Take-profit exits
      - Max drawdown circuit-breaker
      - Partial position sizing
    """

    def __init__(self, df: pd.DataFrame, signals: pd.Series,
                 strategy_name: str = "Strategy",
                 initial_cash: float = 100_000,
                 commission: float = COMMISSION,
                 slippage: float = SLIPPAGE,
                 atr_sl_mult: float = 1.5,
                 atr_tp_mult: float = 3.0,
                 max_dd_gate: float = MAX_DRAWDOWN_GATE):

        self.df       = df.copy()
        self.signals  = signals.reindex(df.index).fillna(0)
        self.name     = strategy_name
        self.cash     = initial_cash
        self.init_cash= initial_cash
        self.comm     = commission
        self.slip     = slippage
        self.sl_mult  = atr_sl_mult
        self.tp_mult  = atr_tp_mult
        self.dd_gate  = max_dd_gate

        self.position     = 0.0
        self.entry_price  = 0.0
        self.stop_loss    = 0.0
        self.take_profit  = 0.0
        self.portfolio    = []
        self.trades       = []
        self.peak         = initial_cash
        self.halted       = False

    def _fill_price(self, close: float, side: str) -> float:
        """Apply slippage: adverse for both buy and sell."""
        return close * (1 + self.slip) if side == "BUY" else close * (1 - self.slip)

    def _buy(self, idx, price, atr):
        if self.cash <= 0:
            return
        fp   = self._fill_price(price, "BUY")
        size = (self.cash * RISK_PER_TRADE * 5) // fp    # ~10% of portfolio
        size = max(1, size)
        cost = size * fp * (1 + self.comm)
        if cost > self.cash:
            size = int(self.cash / (fp * (1 + self.comm)))
        if size <= 0:
            return
        self.cash       -= size * fp * (1 + self.comm)
        self.position    = size
        self.entry_price = fp
        self.stop_loss   = fp - self.sl_mult * atr
        self.take_profit = fp + self.tp_mult * atr
        self.trades.append({"type": "BUY",  "idx": idx, "price": round(fp, 4), "size": size, "atr": round(atr, 4)})

    def _sell(self, idx, price, reason="signal"):
        if self.position <= 0:
            return
        fp         = self._fill_price(price, "SELL")
        proceeds   = self.position * fp * (1 - self.comm)
        pnl        = proceeds - self.position * self.entry_price
        self.cash += proceeds
        self.trades.append({"type": "SELL", "idx": idx, "price": round(fp, 4),
                             "size": self.position, "pnl": round(pnl, 2), "reason": reason})
        self.position    = 0
        self.entry_price = 0.0

    def run(self) -> BacktestResult:
        df  = self.df
        sig = self.signals

        for i in range(len(df)):
            close = df["Close"].iloc[i]
            atr   = df["ATR"].iloc[i] if "ATR" in df.columns else close * 0.02
            s     = sig.iloc[i]

            # Check drawdown gate
            total = self.cash + self.position * close
            self.peak = max(self.peak, total)
            dd = (total - self.peak) / (self.peak + 1e-9)
            if dd < self.dd_gate and not self.halted:
                self.halted = True          # circuit-breaker

            if self.halted:
                if self.position > 0:
                    self._sell(i, close, "drawdown_gate")
                self.portfolio.append(self.cash + self.position * close)
                continue

            # Stop-loss / take-profit
            if self.position > 0:
                if close <= self.stop_loss:
                    self._sell(i, close, "stop_loss")
                elif close >= self.take_profit:
                    self._sell(i, close, "take_profit")

            # Entry / exit signals
            if s == 1 and self.position == 0:
                self._buy(i, close, atr)
            elif s == -1 and self.position > 0:
                self._sell(i, close, "signal")

            self.portfolio.append(self.cash + self.position * close)

        return self._compute_results()

    def _compute_results(self) -> BacktestResult:
        pv     = pd.Series(self.portfolio)
        rets   = pv.pct_change().dropna()
        final  = pv.iloc[-1] if len(pv) else self.init_cash
        tot_ret= (final / self.init_cash - 1) * 100
        sharpe = float(np.sqrt(252) * rets.mean() / (rets.std() + 1e-9)) if len(rets) > 1 else 0
        dd_ser = pv / pv.cummax() - 1
        max_dd = float(dd_ser.min() * 100)

        sell_trades = [t for t in self.trades if t["type"] == "SELL" and "pnl" in t]
        wins        = [t for t in sell_trades if t["pnl"] > 0]
        losses      = [t for t in sell_trades if t["pnl"] <= 0]
        win_rate    = len(wins) / max(len(sell_trades), 1) * 100
        gross_profit= sum(t["pnl"] for t in wins)
        gross_loss  = abs(sum(t["pnl"] for t in losses))
        pf          = gross_profit / (gross_loss + 1e-9)
        calmar      = tot_ret / (abs(max_dd) + 1e-9)

        return BacktestResult(
            strategy_name    = self.name,
            final_value      = round(final, 2),
            total_return_pct = round(tot_ret, 2),
            sharpe_ratio     = round(sharpe, 3),
            max_drawdown_pct = round(max_dd, 2),
            win_rate_pct     = round(win_rate, 1),
            total_trades     = len([t for t in self.trades if t["type"] == "BUY"]),
            profit_factor    = round(pf, 3),
            calmar_ratio     = round(calmar, 3),
            trades           = self.trades,
            portfolio_series = self.portfolio,
        )



### Signal Generation

In [14]:
class SignalEngine:
    """
    Aggregates signals from top-ranked strategies.
    FIXES APPLIED:
      1. State-aware consensus: tracks open positions, not just crossover bars
      2. Revenue growth unit normalisation (yfinance gives 0.145, not 14.5)
      3. Sector-aware D/E thresholds (banks have 600+ D/E by convention)
      4. Lowered BUY/SELL thresholds (0.10 vs 0.25) — reachable with 3 strategies
      5. ADX classifier threshold lowered from 20 → 15
    """

    def __init__(self, df: pd.DataFrame, strategies: List[StrategyResult],
                 backtests: List[BacktestResult], fundamentals: Dict,
                 regime: Regime, regime_confidence: float):
        self.df           = df
        self.strategies   = strategies
        self.backtests    = backtests
        self.fundamentals = fundamentals
        self.regime       = regime
        self.regime_conf  = regime_confidence

    def _fundamental_score(self) -> float:
        """Score 0-1.  FIX: normalise yfinance decimal→% and sector D/E."""
        f     = self.fundamentals
        score = 0.5
        if not f:
            return score

        pe  = f.get("pe_ratio", 20)
        rev = f.get("revenue_growth", 0)
        de  = f.get("debt_equity", 0.5)
        div = f.get("dividend_yield", 0)

        # FIX 2: yfinance returns revenue_growth as decimal (0.145 = 14.5%)
        if rev is not None and abs(rev) < 5:   # clearly a decimal, not a %
            rev = rev * 100

        # FIX 3: sector-aware D/E — banks legitimately have D/E 500-1000
        de_threshold = 1500 if (de is not None and de > 100) else 2.0

        score += 0.15 if pe < 20 else (-0.1 if pe > 40 else 0)
        score += 0.15 if rev > 15 else (-0.1 if rev < 0 else 0)
        score -= 0.10 if (de is not None and de > de_threshold) else 0
        score += 0.10 if (div is not None and div > 0.02) else 0  # FIX: yfinance div as decimal too
        return max(0, min(1, score))

    @staticmethod
    def _make_position_signal(signals: pd.Series) -> pd.Series:
        """
        FIX 1 — State-aware signal.
        Convert event-based crossover signals (+1/-1 only on crossover bars)
        into a POSITION signal: +1 while in a long trade, -1 while in a short.
        This means the LAST bar reflects current regime, not just today's event.
        """
        position = 0
        pos_series = []
        for val in signals:
            if val == 1:
                position = 1
            elif val == -1:
                position = -1 if position == 1 else 0  # exit long → flat, or go short
            pos_series.append(position)
        return pd.Series(pos_series, index=signals.index)

    def generate(self) -> Tuple[Signal, float, List[str]]:
        last_idx = self.df.index[-1]
        bt_map   = {bt.strategy_name: bt for bt in self.backtests}

        weighted_sum = 0.0
        total_weight = 0.0
        reasons      = []
        per_strat    = []

        for strat in self.strategies:
            # FIX 1: use position-state signal, not raw crossover signal
            pos_sig = self._make_position_signal(strat.signals)
            sig_val = pos_sig.get(last_idx, 0)

            bt      = bt_map.get(strat.name)
            # FIX: use abs Sharpe so good-bear strategies still count; floor at 0.1
            sharpe  = bt.sharpe_ratio if bt else 0.0
            weight  = max(sharpe, 0.1)
            weighted_sum += sig_val * weight
            total_weight += weight
            per_strat.append(f"{strat.name[:28]}: pos={sig_val:+d}, w={weight:.2f}")

        consensus = weighted_sum / (total_weight + 1e-9)
        fund_score = self._fundamental_score()

        # Build reasoning
        row = self.df.iloc[-1]
        reasons.append(f"Regime: {self.regime.value} (confidence {self.regime_conf:.0%})")
        reasons.append(f"Fundamental score: {fund_score:.2f} / 1.00")
        reasons.append(f"Weighted position consensus: {consensus:+.3f}")
        reasons.append(f"RSI: {row.get('RSI', 0):.1f}  |  ADX: {row.get('ADX', 0):.1f}")
        reasons.append(f"MACD Hist: {row.get('MACD_Hist', 0):+.4f}")
        reasons.append(f"Price vs EMA20: {((row['Close']/row.get('EMA_20',row['Close']))-1)*100:+.2f}%")
        for s in per_strat:
            reasons.append(f"  → {s}")

        # FIX 4: lowered thresholds — 0.10 is reachable with 3 strategies
        BUY_THRESH  =  0.10   # was 0.25
        SELL_THRESH = -0.10   # was -0.25
        MIN_CONF    =  0.40

        if self.regime_conf < MIN_CONF:
            reasons.append("⚠ Regime confidence too low — HOLD.")
            return Signal.HOLD, 0.0, reasons

        if consensus >= BUY_THRESH and fund_score >= 0.40:
            confidence = min(abs(consensus) * fund_score * self.regime_conf, 1.0)
            reasons.append(f"✓ BUY signal (consensus {consensus:+.3f}, fund {fund_score:.2f})")
            return Signal.BUY, round(confidence, 3), reasons

        if consensus <= SELL_THRESH:
            confidence = min(abs(consensus) * (1 - fund_score + 0.2) * self.regime_conf, 1.0)
            reasons.append(f"✓ SELL signal (consensus {consensus:+.3f})")
            return Signal.SELL, round(confidence, 3), reasons

        reasons.append("Signal too weak or conflicting — HOLD.")
        return Signal.HOLD, 0.0, reasons



### Reporter

In [15]:
class Reporter:
    """Renders a full terminal report."""

    SEP  = "─" * 68
    DSEP = "═" * 68

    @staticmethod
    def _bar(val, max_val=100, width=20, char="█", empty="░") -> str:
        filled = int(width * min(val, max_val) / (max_val + 1e-9))
        return char * filled + empty * (width - filled)

    def print_header(self, ticker: str, data_summary: Dict):
        print(f"\n{self.DSEP}")
        print(f"  QUANTITATIVE TRADING STRATEGY ENGINE")
        print(f"  Ticker: {ticker}  |  {data_summary['start']} → {data_summary['end']}")
        print(f"  Rows: {data_summary['rows']}  |  Last Close: ${data_summary['price_last']}")
        print(f"{self.DSEP}")

    def print_regime(self, regime: Regime, confidence: float, detail: Dict):
        icons = {Regime.UP: "▲", Regime.DOWN: "▼", Regime.SIDEWAYS: "◆"}
        colors= {Regime.UP: "BULLISH", Regime.DOWN: "BEARISH", Regime.SIDEWAYS: "NEUTRAL"}
        print(f"\n{'TREND REGIME':─<68}")
        print(f"  Detected: {icons[regime]} {regime.value} ({colors[regime]})  "
              f"Confidence: {confidence:.0%}  {self._bar(confidence*100, 100, 15)}")
        print()
        for k, v in detail.items():
            print(f"  {k:<20}: {v}")

    def print_strategies(self, ranked: List[StrategyResult],
                         backtests: List[BacktestResult]):
        bt_map = {bt.strategy_name: bt for bt in backtests}
        print(f"\n{'TOP STRATEGIES FOR REGIME':─<68}")
        for i, s in enumerate(ranked, 1):
            bt = bt_map.get(s.name)
            print(f"\n  [{i}] {s.name}")
            print(f"      {s.description}")
            print(f"      Params: {s.param_summary}")
            if bt:
                ret_symbol = "+" if bt.total_return_pct >= 0 else ""
                dd_symbol  = ""
                print(f"      Backtest → Return: {ret_symbol}{bt.total_return_pct:.2f}%  "
                      f"Sharpe: {bt.sharpe_ratio:.3f}  "
                      f"MaxDD: {bt.max_drawdown_pct:.2f}%  "
                      f"WinRate: {bt.win_rate_pct:.1f}%  "
                      f"PF: {bt.profit_factor:.2f}  "
                      f"Trades: {bt.total_trades}")

    def print_backtest_table(self, backtests: List[BacktestResult]):
        print(f"\n{'FULL BACKTEST COMPARISON':─<68}")
        hdr = f"  {'Strategy':<34} {'Return%':>8} {'Sharpe':>7} {'MaxDD%':>7} {'WinRate':>8} {'PF':>6} {'N':>4}"
        print(hdr)
        print(f"  {'─'*64}")
        for bt in sorted(backtests, key=lambda x: x.sharpe_ratio, reverse=True):
            flag = "✓" if bt.sharpe_ratio > 0.5 and bt.total_return_pct > 0 else " "
            print(f"  {flag} {bt.strategy_name:<33} "
                  f"{bt.total_return_pct:>+8.2f} "
                  f"{bt.sharpe_ratio:>7.3f} "
                  f"{bt.max_drawdown_pct:>7.2f} "
                  f"{bt.win_rate_pct:>7.1f}% "
                  f"{bt.profit_factor:>6.2f} "
                  f"{bt.total_trades:>4}")

    def print_signal(self, signal: Signal, confidence: float,
                     trade_params: Optional[TradeParams], reasons: List[str]):
        icons = {Signal.BUY: "▲ BUY", Signal.SELL: "▼ SELL", Signal.HOLD: "◆ HOLD"}
        borders= {Signal.BUY: "═", Signal.SELL: "═", Signal.HOLD: "─"}
        b = borders[signal]
        print(f"\n{b*68}")
        print(f"  FINAL SIGNAL: {icons[signal]}")
        if signal != Signal.HOLD:
            print(f"  Confidence:   {confidence:.1%}  {self._bar(confidence*100, 100, 20)}")
        print(f"{b*68}")
        print(f"\n  Reasoning:")
        for r in reasons:
            print(f"    • {r}")

        if trade_params and signal != Signal.HOLD:
            print(f"\n  Trade Parameters:")
            print(f"    Entry Price   : ${trade_params.entry_price}")
            print(f"    Stop-Loss     : ${trade_params.stop_loss}  "
                  f"({abs(trade_params.entry_price - trade_params.stop_loss)/trade_params.entry_price*100:.2f}% risk)")
            print(f"    Take-Profit   : ${trade_params.take_profit}  "
                  f"({abs(trade_params.take_profit - trade_params.entry_price)/trade_params.entry_price*100:.2f}% target)")
            print(f"    Risk/Reward   : 1 : {trade_params.risk_reward:.2f}")
            print(f"    Position Size : {trade_params.position_size:.1%} of portfolio")



### Quant Engine

In [16]:
class QuantTradingEngine:
    """
    Orchestrates all components end-to-end.

    Usage:
        engine = QuantTradingEngine(price_df, fundamentals, ticker="AAPL")
        result = engine.run()
    """

    def __init__(self, price_df: pd.DataFrame,
                 fundamentals: Optional[Dict] = None,
                 ticker: str = "ASSET",
                 initial_cash: float = 100_000):
        self.price_df     = price_df
        self.fundamentals = fundamentals or {}
        self.ticker       = ticker
        self.initial_cash = initial_cash

    def run(self) -> Dict:
        reporter = Reporter()

        # ── 1. Load & validate ────────────────────────────────────────────────
        data_engine = DataEngine(self.price_df, self.fundamentals).load()
        data_summary= data_engine.summary()

        # ── 2. Feature engineering ────────────────────────────────────────────
        df = FeatureEngine(data_engine.get()).build_all()

        reporter.print_header(self.ticker, data_summary)

        # ── 3. Trend classification ───────────────────────────────────────────
        regime, confidence, regime_detail = TrendClassifier(df).classify()
        reporter.print_regime(regime, confidence, regime_detail)

        # ── 4. Build all strategies ───────────────────────────────────────────
        all_strategies = StrategyLibrary(df).get_all()

        # ── 5. Select top strategies for regime ───────────────────────────────
        selector = StrategySelector(df, regime, confidence)
        top_strats = selector.rank(all_strategies, top_k=3)

        # ── 6. Backtest ALL strategies ────────────────────────────────────────
        all_backtests = []
        for s in all_strategies:
            bt = Backtester(df, s.signals, s.name,
                            initial_cash=self.initial_cash).run()
            all_backtests.append(bt)

        reporter.print_strategies(top_strats, all_backtests)
        reporter.print_backtest_table(all_backtests)

        # ── 7. Risk management ────────────────────────────────────────────────
        risk_mgr = RiskManager(df, regime, self.initial_cash)

        # ── 8. Signal generation ──────────────────────────────────────────────
        top_bt = [bt for bt in all_backtests
                  if bt.strategy_name in [s.name for s in top_strats]]

        signal_engine = SignalEngine(df, top_strats, top_bt,
                                     self.fundamentals, regime, confidence)
        signal, sig_confidence, reasons = signal_engine.generate()

        trade_params = risk_mgr.compute_trade_params(signal)
        reporter.print_signal(signal, sig_confidence, trade_params, reasons)

        print(f"\n{Reporter.DSEP}\n")

        # ── Return machine-readable result ────────────────────────────────────
        return {
            "ticker":           self.ticker,
            "regime":           regime.value,
            "regime_confidence":confidence,
            "signal":           signal.value,
            "signal_confidence":sig_confidence,
            "trade_params":     trade_params.__dict__ if trade_params else None,
            "top_strategies":   [s.name for s in top_strats],
            "backtests":        {bt.strategy_name: {
                                    "return_pct":   bt.total_return_pct,
                                    "sharpe":       bt.sharpe_ratio,
                                    "max_dd_pct":   bt.max_drawdown_pct,
                                    "win_rate_pct": bt.win_rate_pct,
                                    "profit_factor":bt.profit_factor,
                                    "trades":       bt.total_trades,
                                } for bt in all_backtests},
            "reasons":          reasons,
        }


### Quant Engine Simulator

In [31]:
comp="PLTR"
data=pd.DataFrame()
data=data.ta.ticker(comp,period="1y",interval="1h")

In [32]:
import yfinance as yf

def get_fundamental_data(ticker_symbol: str) -> dict:
    ticker = yf.Ticker(ticker_symbol)
    info = ticker.info

    fundamentals = {
        "pe_ratio":       info.get("trailingPE"),
        "revenue_growth": info.get("revenueGrowth"),   # decimal e.g. 0.145
        "debt_equity":    info.get("debtToEquity"),
        "dividend_yield": info.get("dividendYield"),   # decimal e.g. 0.02
        "sector":         info.get("sector", ""),
    }
    result = {k: v for k, v in fundamentals.items() if v is not None}
    print(f"Fundamentals fetched: {result}")
    return result

fundamentals = get_fundamental_data(comp)
print(fundamentals)


Fundamentals fetched: {'pe_ratio': 232.36508, 'revenue_growth': 0.7, 'debt_equity': 3.063, 'sector': 'Technology'}
{'pe_ratio': 232.36508, 'revenue_growth': 0.7, 'debt_equity': 3.063, 'sector': 'Technology'}


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


In [33]:
engine = QuantTradingEngine(
        price_df     = data,
        fundamentals = fundamentals,
        ticker       = comp,
        initial_cash = 100_000,
    )
result = engine.run()


════════════════════════════════════════════════════════════════════
  QUANTITATIVE TRADING STRATEGY ENGINE
  Ticker: PLTR  |  2025-04-21 → 2026-04-17
  Rows: 1725  |  Last Close: $146.37
════════════════════════════════════════════════════════════════════

TREND REGIME────────────────────────────────────────────────────────
  Detected: ◆ SIDEWAYS (NEUTRAL)  Confidence: 60%  ████████░░░░░░░

  MA Slope            : SIDEWAYS
  ADX/DI              : UPTREND
  BB Width            : SIDEWAYS
  Price vs MA         : UPTREND
  Linear Reg          : SIDEWAYS
  Votes UP            : 2
  Votes DOWN          : 0
  Votes SIDEWAYS      : 3

TOP STRATEGIES FOR REGIME───────────────────────────────────────────

  [1] Bollinger Band Reversion
      Mean-reversion on BB touch, only in low-bandwidth regime.
      Params: BB(20,2), BB Width < avg, RSI filter
      Backtest → Return: +0.44%  Sharpe: 0.092  MaxDD: -2.53%  WinRate: 35.3%  PF: 1.18  Trades: 17

  [2] Stochastic Swing
      Stochastic %K/%D